# 3. YOLO Training
Questo notebook configura il training del modello YOLO.

In [5]:
import os, yaml, urllib.request, subprocess
from pathlib import Path
subprocess.run(['pip', 'install', '-q', 'ultralytics==8.3.86', 'numpy<2'], check=True)
from ultralytics import YOLO

In [6]:
# ── Struttura Cartelle ────────────────────────────────────────────────────────
BASE = Path('/teamspace/studios/this_studio/project-sbs')
DATASET_YOLO = BASE / 'datasets' / 'dataset_yolo'
WEIGHTS_DIR = BASE / 'model_weights' / 'yolo'
RUNS_DIR = BASE / 'runs'
RUNS_WEIGHTS_DIR = RUNS_DIR / 'yolo_banner_training' / 'weights'
YAML_PATH = DATASET_YOLO / 'data.yaml'
YOLO_START_W = WEIGHTS_DIR / 'yolo12n.pt'

for d in [WEIGHTS_DIR, RUNS_DIR, RUNS_WEIGHTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [7]:
# ── Configurazione YAML ───────────────────────────────────────────────────────
yaml_content = {
    'path': str(DATASET_YOLO),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 1,
    'names': ['banner']
}
YAML_PATH.write_text(yaml.dump(yaml_content, default_flow_style=False), encoding='utf-8')
print(f'✅ YAML scritto in {YAML_PATH}')

✅ YAML scritto in /teamspace/studios/this_studio/project-sbs/datasets/dataset_yolo/data.yaml


In [8]:
# ── Download Pesi e Training ───────────────────────────────────────────────────
import shutil
if not YOLO_START_W.exists():
    print('⬇️ Download automatico tramite Ultralytics...')
    # Passando il nome base, YOLO scaricherà in automatico l'ultima versione dal GitHub corretto
    temp_model = YOLO('yolo12n.pt')

    # Cerchiamo dove ha salvato il file e lo copiamo nella nostra cartella personalizzata
    downloaded = None
    if hasattr(temp_model, 'ckpt_path') and temp_model.ckpt_path:
        downloaded = Path(temp_model.ckpt_path)
    elif Path('yolo12n.pt').exists():
        downloaded = Path('yolo12n.pt')

    if downloaded and downloaded.exists():
        shutil.move(str(downloaded), str(YOLO_START_W))
        print(f'✅ Pesi clonati con successo su {YOLO_START_W}')
    else:
        print('⚠️ Attenzione: Pesi base scaricati ma non rintracciabili per lo spostamento.')
else:
    print(f'✅ Pesi base trovati in {YOLO_START_W}')

run_dir = RUNS_DIR / 'yolo_banner_training'
if (run_dir / 'weights' / 'best.pt').exists():
    print('✅ Training già completato con best.pt esistente.')
else:
    print('🚀 Avvio Training YOLO...')
    # Carica in memoria i pesi dal path effettivo che desideri tu
    model = YOLO(str(YOLO_START_W))
    results = model.train(
        data=str(YAML_PATH), project=str(RUNS_DIR), name='yolo_banner_training',
        epochs=30, imgsz=640, batch=64, workers=16, rect=True, multi_scale=True,
        cache='disk', patience=15, save_period=10, exist_ok=True,
        hsv_h=0.02, hsv_s=0.8, hsv_v=0.4, degrees=10, translate=0.2, scale=0.3
    )
    print('✅ Training completato!')

⬇️ Download automatico tramite Ultralytics...


100%|██████████| 5.34M/5.34M [00:00<00:00, 63.9MB/s]

✅ Pesi clonati con successo su /teamspace/studios/this_studio/project-sbs/model_weights/yolo/yolo12n.pt
✅ Training già completato con best.pt esistente.
